<a href="https://colab.research.google.com/github/yanyanone/Calc/blob/main/Activity_2_MobileNetV2_Leaf_Disease_Scanner_(Colab).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Activity 2: Image Processing — Leaf Disease Scanner 🍃
**STIWNU Computing Seminar • November 23, 2026**  
**Time:** ~35 minutes

> ⚠️ **Do this first:** `Runtime → Change runtime type → T4 GPU → Save`, then run the cells. (Training is ~10× slower on CPU.)

### How does a computer "see" a photo?
To a computer, an image is just a **grid of numbers**. Each pixel has 3 values (**R**ed, **G**reen, **B**lue) from **0 to 255**. Image processing means **doing math on those numbers**, and deep learning means letting a model **learn which patterns of numbers** mean "diseased leaf".

### What you will do
| Part | Steps | Key concept |
|---|---|---|
| **A. Image processing basics** | 1–5 | View images, pixels, color channels, transformations |
| **B. Prepare data for AI** | 6–7 | Resizing, batching, data augmentation |
| **C. Train a classifier** | 8–9 | Transfer learning with MobileNetV2 |
| **D. Evaluate & predict** | 10–12 | Learning curves, confusion matrix, predictions on new images |
| **E. Deploy (bonus)** | 13 | Mobile model + web app |

> **How to use this notebook:** Run each cell **from top to bottom** with `Shift + Enter`. Read the text cell *before* each code cell; it explains what's about to happen.
>
> 💬 **Think**: a question to discuss with your seatmate &nbsp;•&nbsp; 🧪 **Try it**: change a value and re-run &nbsp;•&nbsp; 💡 **Key idea**: the takeaway to remember

In [ ]:
# ==============================================================================
# SETUP: Install & import libraries, check for GPU
# ==============================================================================
!pip install -q gradio datasets

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from datasets import load_dataset
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix

print('TensorFlow version:', tf.__version__)
if tf.config.list_physical_devices('GPU'):
    print('✅ GPU is ON. Training will be fast.')
else:
    print('⚠️ No GPU. Go to Runtime → Change runtime type → T4 GPU (training will be slow on CPU).')

# Part A: Image Processing Basics

## Step 1: Load the Dataset
We use the **`beans`** dataset: real smartphone photos of bean leaves taken **outdoors on farms in Uganda**, with messy backgrounds and natural lighting, much like what you'd capture on a Philippine farm.

| Class | What it looks like |
|---|---|
| `angular_leaf_spot` | Brown, angular (straight-edged) spots, limited by leaf veins |
| `bean_rust` | Small, round, reddish-brown "rusty" dots |
| `healthy` | Uniform green leaf |

The dataset is already split into **train / validation / test** sets. We download it from the **Hugging Face Hub**, a free library of public AI datasets.

In [ ]:
# ==============================================================================
# STEP 1: Download the beans dataset (~180 MB, about 1 minute)
# ==============================================================================
beans = load_dataset('AI-Lab-Makerere/beans')   # downloads train / validation / test splits

CLASS_NAMES = beans['train'].features['labels'].names
NUM_CLASSES = len(CLASS_NAMES)

def to_tf_dataset(split):
    # Wrap a Hugging Face split as a TensorFlow dataset of (image, label) pairs
    def generator():
        for example in beans[split]:
            yield np.array(example['image'].convert('RGB')), example['labels']
    return tf.data.Dataset.from_generator(generator, output_signature=(
        tf.TensorSpec(shape=(None, None, 3), dtype=tf.uint8),
        tf.TensorSpec(shape=(), dtype=tf.int64))
    ).apply(tf.data.experimental.assert_cardinality(len(beans[split])))   # tell TF how many images there are

raw_train, raw_val, raw_test = (to_tf_dataset(s) for s in ['train', 'validation', 'test'])

print('Classes:', CLASS_NAMES)
for split in ['train', 'validation', 'test']:
    print(f'  {split:10s}: {len(beans[split])} images')

## Step 2: View the First 2 Images in the Dataset 👀
Always **look at your images** before training. Check the lighting, backgrounds, image size, and whether the labels look right.

In [ ]:
# ==============================================================================
# STEP 2: Display the first 2 training images with their labels
# ==============================================================================
first_two = [(img.numpy(), int(lbl.numpy())) for img, lbl in raw_train.take(2)]

plt.figure(figsize=(12, 6))
for i, (img, lbl) in enumerate(first_two):
    plt.subplot(1, 2, i + 1)
    plt.imshow(img)
    plt.title(f'Image #{i + 1}  •  Label: {CLASS_NAMES[lbl]}\nSize: {img.shape[1]}×{img.shape[0]} px, {img.shape[2]} color channels')
    plt.axis('off')
plt.tight_layout()
plt.show()

sample_img = first_two[0][0]   # we'll use image #1 for the next steps

## Step 3: An Image Is Just Numbers 🔢
Let's prove it. We'll print the image's **shape**, the **range of pixel values**, and **zoom into a tiny 10×10 patch** to see the actual numbers.

In [ ]:
# ==============================================================================
# STEP 3: Look at the raw pixel numbers
# ==============================================================================
print('Shape (height, width, channels):', sample_img.shape)
print('Data type:', sample_img.dtype, '| min value:', sample_img.min(), '| max value:', sample_img.max())
print('Total numbers in this one image:', f'{sample_img.size:,}')
print('\nPixel at row 250, column 250 → [R, G, B] =', sample_img[250, 250])

r0, c0 = 160, 300                     # 🧪 try another location (row, column)!
patch = sample_img[r0:r0 + 10, c0:c0 + 10]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(sample_img)
axes[0].add_patch(plt.Rectangle((c0, r0), 10, 10, edgecolor='red', facecolor='none', linewidth=2))
axes[0].set_title('Full image (red box = zoomed patch)')
axes[1].imshow(patch, interpolation='nearest')
for r in range(10):
    for c in range(10):
        axes[1].text(c, r, patch[r, c, 1], ha='center', va='center', fontsize=8, color='white')
axes[1].set_title('Zoomed 10×10 patch (numbers = GREEN value of each pixel)')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

## Step 4: Color Channels, Grayscale & Histograms 🎨
A color image is really **3 grayscale images stacked**: Red, Green, and Blue. Healthy leaves have strong **green** values; disease spots usually add **red/brown**.

A **histogram** counts how many pixels have each brightness value, which makes it a quick "fingerprint" of the image's colors.

In [ ]:
# ==============================================================================
# STEP 4: Split into R, G, B channels + grayscale + color histogram
# ==============================================================================
fig, axes = plt.subplots(1, 5, figsize=(20, 4.5))
axes[0].imshow(sample_img); axes[0].set_title('Original (RGB)')
for c, (name, cmap) in enumerate([('Red', 'Reds_r'), ('Green', 'Greens_r'), ('Blue', 'Blues_r')]):
    axes[c + 1].imshow(sample_img[:, :, c], cmap=cmap)
    axes[c + 1].set_title(f'{name} channel only')
gray = tf.image.rgb_to_grayscale(sample_img).numpy().squeeze()
axes[4].imshow(gray, cmap='gray'); axes[4].set_title('Grayscale (1 channel)')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 3.5))
for c, color in enumerate(['red', 'green', 'blue']):
    plt.hist(sample_img[:, :, c].ravel(), bins=64, color=color, alpha=0.5, label=color)
plt.title('Color histogram of image #1')
plt.xlabel('Pixel value (0 = dark, 255 = bright)')
plt.ylabel('Number of pixels')
plt.legend()
plt.show()

## Step 5: Common Image Transformations 🔄
These basic operations are the building blocks of image processing. We'll use several of them later as **data augmentation**.

| Operation | Why it's useful |
|---|---|
| **Resize** | AI models need every image to be the **same size** |
| **Flip / Rotate** | A leaf is still a leaf when the phone is held differently |
| **Brightness / Contrast** | Simulates sunny vs. cloudy days |
| **Crop** | Focus on the important region |

In [ ]:
# ==============================================================================
# STEP 5: Apply basic transformations to image #1
# ==============================================================================
img_f = tf.image.convert_image_dtype(sample_img, tf.float32)   # scale 0–255 → 0.0–1.0

transforms = {
    'Original':            img_f,
    'Resized to 32×32':    tf.image.resize(img_f, (32, 32)),
    'Resized to 224×224':  tf.image.resize(img_f, (224, 224)),
    'Flipped left-right':  tf.image.flip_left_right(img_f),
    'Rotated 90°':         tf.image.rot90(img_f),
    'Brighter (+0.3)':     tf.image.adjust_brightness(img_f, 0.3),
    'Higher contrast (×2)': tf.image.adjust_contrast(img_f, 2.0),
    'Center crop (50%)':   tf.image.central_crop(img_f, 0.5),
}

plt.figure(figsize=(16, 8))
for i, (name, t) in enumerate(transforms.items()):
    plt.subplot(2, 4, i + 1)
    plt.imshow(tf.clip_by_value(t, 0, 1), interpolation='nearest')
    plt.title(f'{name}\n{t.shape[1]}×{t.shape[0]} px')
    plt.axis('off')
plt.tight_layout()
plt.show()

💬 **Think:**
- At **32×32**, can you still tell if the leaf is diseased? What detail is lost when we shrink images too much?
- Flipping a leaf doesn't change its disease. Can you think of an image task where flipping **would** change the answer? *(Hint: the digits 6 and 9, or road signs with arrows.)*

# Part B: Prepare the Data for AI

## Step 6: See Several Examples of Each Class
Before training, make sure **you** can see the differences between classes. If a human can't tell them apart, a model will struggle too.

In [ ]:
# ==============================================================================
# STEP 6: A gallery of 4 examples per class
# ==============================================================================
per_class = {c: [] for c in range(NUM_CLASSES)}
for img, lbl in raw_train:
    l = int(lbl.numpy())
    if len(per_class[l]) < 4:
        per_class[l].append(img.numpy())
    if all(len(v) == 4 for v in per_class.values()):
        break

plt.figure(figsize=(14, 10))
for row, cls in enumerate(per_class):
    for col, img in enumerate(per_class[cls]):
        plt.subplot(NUM_CLASSES, 4, row * 4 + col + 1)
        plt.imshow(img)
        plt.axis('off')
        if col == 0:
            plt.title(CLASS_NAMES[cls], loc='left', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 7: Build the Data Pipeline & Data Augmentation
**Pipeline:** every image is **resized to 224×224** (the size MobileNetV2 expects) and grouped into **batches of 32** (the model learns from 32 images at a time).

**Data augmentation:** we only have ~1,000 training photos. To help the model generalize, we create **random variations** (flip, rotate, zoom, contrast) **on the fly** during training. Every epoch, the model sees slightly different versions of each photo.

> ⚠️ Augmentation is applied **only to training images**, never to validation or test images. Those must stay "real".

In [ ]:
# ==============================================================================
# STEP 7a: Resize, shuffle, batch, and prefetch
# ==============================================================================
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def resize_image(image, label):
    return tf.image.resize(image, (IMG_SIZE, IMG_SIZE)), label

train_ds = raw_train.map(resize_image, num_parallel_calls=AUTOTUNE).cache().shuffle(1000, seed=42).batch(BATCH_SIZE).prefetch(AUTOTUNE)
val_ds   = raw_val.map(resize_image, num_parallel_calls=AUTOTUNE).cache().batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds  = raw_test.map(resize_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

images, labels = next(iter(train_ds))
print('One batch of images has shape:', images.shape, '→ (batch size, height, width, channels)')
print('Labels in this batch:', labels.numpy())

In [ ]:
# ==============================================================================
# STEP 7b: Data augmentation, the same leaf shown 8 random ways
# ==============================================================================
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.15),
], name='augmentation')

leaf = tf.image.resize(sample_img, (IMG_SIZE, IMG_SIZE))[tf.newaxis, ...]
plt.figure(figsize=(16, 4.5))
for i in range(8):
    plt.subplot(1, 8, i + 1)
    shown = leaf[0] if i == 0 else data_augmentation(leaf, training=True)[0]
    plt.imshow(np.clip(shown.numpy() / 255.0, 0, 1))
    plt.title('Original' if i == 0 else f'Random #{i}')
    plt.axis('off')
plt.tight_layout()
plt.show()

# Part C: Train an Image Classifier

## Step 8: Transfer Learning with MobileNetV2 🧠
Training an image model **from scratch** needs millions of photos. Instead, we use **transfer learning**:

> 🎓 **Analogy:** It's easier to teach plant diseases to someone who **already knows how to see** (edges, textures, shapes) than to a newborn.

**MobileNetV2** was already trained on **1.4 million images** (ImageNet). We:
1. ❄️ **Freeze** its layers: keep its knowledge of edges, textures, and shapes.
2. ➕ **Add a small new "head"** that learns our 3 leaf classes.

```
Leaf photo → [augmentation] → [MobileNetV2 (frozen)] → 1,280 features → [new Dense layer] → 3 probabilities
```
MobileNetV2 is designed for **phones**: small and fast, ideal for a farm app.

In [ ]:
# ==============================================================================
# STEP 8: Build the transfer-learning model
# ==============================================================================
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,        # remove the original 1,000-class ImageNet head
    weights='imagenet'        # load the pre-trained knowledge
)
base_model.trainable = False  # ❄️ freeze it

inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)                               # only active during training
x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1.0)(x)  # pixels 0–255 → -1…1 (what MobileNetV2 expects)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)             # summarize each feature map into 1 number
x = tf.keras.layers.Dropout(0.3)(x)                         # randomly drop 30% to reduce overfitting
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)   # 3 probabilities that sum to 1

model = tf.keras.Model(inputs, outputs, name='LeafScanner')
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

count = lambda ws: sum(int(np.prod(w.shape)) for w in ws)
print(f'Total parameters     : {count(model.weights):,}')
print(f'❄️ Frozen (pre-trained): {count(model.non_trainable_weights):,}')
print(f'🔥 Trainable (new head): {count(model.trainable_weights):,}  ← only these are learned now!')

## Step 9: Train the Model 🏋️
- **Epoch** = one full pass through all training images.
- **loss** ↓ = how wrong the model is (lower is better). **accuracy** ↑ = % correct.
- `val_` numbers are measured on the **validation set** (images the model doesn't train on).
- **EarlyStopping** stops automatically when validation loss stops improving and keeps the best version.

⏱️ About 1–2 minutes on a T4 GPU.

In [ ]:
# ==============================================================================
# STEP 9a: Train the new head (feature extraction)
# ==============================================================================
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(train_ds, validation_data=val_ds, epochs=8, callbacks=[early_stop])

**Optional, Step 9b: Fine-tuning.** Now we "unfreeze" the top 25 layers of MobileNetV2 and train them **very gently** (learning rate 100× smaller) so they adapt to leaf textures. You can skip this cell if time is short.

In [ ]:
# ==============================================================================
# STEP 9b (OPTIONAL): Fine-tune the top 25 layers of MobileNetV2
# ==============================================================================
base_model.trainable = True
for layer in base_model.layers[:-25]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),     # tiny learning rate protects pre-trained knowledge
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history_fine = model.fit(train_ds, validation_data=val_ds, epochs=4, callbacks=[early_stop])

# Part D: Evaluate & Predict

## Step 10: Learning Curves
- ✅ **Good:** training and validation curves go in the same direction and stay close.
- ⚠️ **Overfitting:** training accuracy keeps rising while validation accuracy flattens or drops. The model is **memorizing** instead of learning.

In [ ]:
# ==============================================================================
# STEP 10: Plot training vs. validation curves
# ==============================================================================
def combined(key):
    fine = history_fine.history.get(key, []) if 'history_fine' in globals() else []
    return history.history[key] + fine

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, metric in zip(axes, ['accuracy', 'loss']):
    ax.plot(combined(metric), 'o-', label=f'Training {metric}')
    ax.plot(combined('val_' + metric), 's-', label=f'Validation {metric}')
    if 'history_fine' in globals():
        ax.axvline(len(history.history[metric]) - 0.5, color='gray', linestyle='--', label='Fine-tuning starts')
    ax.set_xlabel('Epoch'); ax.set_title(f'{metric.title()} per epoch'); ax.legend()
plt.tight_layout()
plt.show()

## Step 11: The Final Exam: Test Set Evaluation
The test set has **never** been seen during training or tuning, so this is the honest score you'd report in a thesis.

In [ ]:
# ==============================================================================
# STEP 11: Evaluate on the untouched test set
# ==============================================================================
test_images, y_true, y_prob = [], [], []
for imgs, lbls in test_ds:
    y_prob.append(model.predict(imgs, verbose=0))
    test_images.append(imgs.numpy().astype('uint8'))
    y_true.append(lbls.numpy())
test_images, y_true, y_prob = np.concatenate(test_images), np.concatenate(y_true), np.concatenate(y_prob)
y_pred = y_prob.argmax(axis=1)

print(f'🎯 TEST ACCURACY: {(y_pred == y_true).mean():.2%}  ({(y_pred == y_true).sum()} of {len(y_true)} images correct)\n')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion matrix (test set)')
plt.tight_layout()
plt.show()

## Step 12: See the Predictions 🔍
Numbers are great, but let's **look**. First the model's answer for the **first 2 test images**, then the images it got **wrong**.

In [ ]:
# ==============================================================================
# STEP 12a: Predictions for the first 2 test images
# ==============================================================================
fig, axes = plt.subplots(2, 2, figsize=(12, 9), gridspec_kw={'width_ratios': [1.3, 1]})
for i in range(2):
    ok = y_pred[i] == y_true[i]
    axes[i, 0].imshow(test_images[i]); axes[i, 0].axis('off')
    axes[i, 0].set_title(f'Test image #{i + 1}  •  Actual: {CLASS_NAMES[y_true[i]]}')
    colors = ['#2E7D32' if c == y_true[i] else '#BDBDBD' for c in range(NUM_CLASSES)]
    axes[i, 1].barh(CLASS_NAMES, y_prob[i], color=colors)
    axes[i, 1].set_xlim(0, 1)
    axes[i, 1].set_title(f"{'CORRECT' if ok else 'WRONG'}: predicted {CLASS_NAMES[y_pred[i]]} ({y_prob[i].max():.0%})")
    axes[i, 1].set_xlabel('Model confidence')
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# STEP 12b: Which images did the model get WRONG?
# ==============================================================================
wrong = np.where(y_pred != y_true)[0]
print(f'The model made {len(wrong)} mistakes out of {len(y_true)} test images.')

if len(wrong):
    n = min(8, len(wrong))
    plt.figure(figsize=(16, 4.5 * ((n + 3) // 4)))
    for j, i in enumerate(wrong[:n]):
        plt.subplot((n + 3) // 4, 4, j + 1)
        plt.imshow(test_images[i]); plt.axis('off')
        plt.title(f'Actual: {CLASS_NAMES[y_true[i]]}\nPredicted: {CLASS_NAMES[y_pred[i]]} ({y_prob[i].max():.0%})', color='darkred')
    plt.tight_layout()
    plt.show()

💬 **Think:** Look at the mistakes. Are they blurry? Is the leaf small, or is the background busy? Is the disease barely visible? **Studying errors tells you what data to collect next.**

# Part E (Bonus): Deploy It 🚀

## Step 13: Export for Phones & Launch a Web App
- `.keras` is the full model, for a web server.
- `.tflite` is a compressed model that runs **offline on an Android phone**, with no internet needed in the field!

> If the venue network blocks the public link, change `share=True` to `share=False`.

In [ ]:
# ==============================================================================
# STEP 13a: Save the model for servers (.keras) and phones (.tflite)
# ==============================================================================
model.save('leaf_scanner.keras')
print(f"💾 Keras model : {os.path.getsize('leaf_scanner.keras') / 1e6:.1f} MB")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]       # compress weights
with open('leaf_scanner.tflite', 'wb') as f:
    f.write(converter.convert())
print(f"📱 TFLite model: {os.path.getsize('leaf_scanner.tflite') / 1e6:.1f} MB  (small enough for a phone!)")

In [ ]:
# ==============================================================================
# STEP 13b: Interactive Leaf Scanner web app (Gradio)
# ==============================================================================
import gradio as gr

os.makedirs('sample_leaves', exist_ok=True)
examples, seen = [], set()
for img, lbl in raw_test:
    c = int(lbl.numpy())
    if c not in seen:
        path = f'sample_leaves/{CLASS_NAMES[c]}.jpg'
        Image.fromarray(img.numpy()).save(path)
        examples.append([path]); seen.add(c)
    if len(seen) == NUM_CLASSES:
        break

ADVICE = {
    'angular_leaf_spot': 'Remove infected leaves, avoid overhead watering, and consult your local DA/MAO office.',
    'bean_rust': 'Improve airflow between plants, sanitize tools, and consider early fungicide treatment.',
    'healthy': 'Leaf looks healthy! Keep monitoring regularly.',
}

def diagnose(pil_image):
    if pil_image is None:
        return {}, 'Please upload a leaf photo.'
    x = np.array(pil_image.convert('RGB').resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32)[np.newaxis]
    p = model.predict(x, verbose=0)[0]
    return {CLASS_NAMES[i].replace('_', ' ').title(): float(p[i]) for i in range(NUM_CLASSES)}, ADVICE[CLASS_NAMES[p.argmax()]]

gr.Interface(
    fn=diagnose,
    inputs=gr.Image(type='pil', label='Upload a bean leaf photo'),
    outputs=[gr.Label(num_top_classes=3, label='Diagnosis'), gr.Textbox(label='Advice')],
    title='🍃 Leaf Disease Scanner (MobileNetV2)',
    examples=examples,
).launch(share=True)

## ✅ Recap
1. **Images are grids of numbers**: height × width × 3 color channels, values 0–255.
2. **Image processing** (resize, flip, rotate, brightness, crop) is math on those numbers.
3. **Data augmentation** creates variety so the model generalizes from few photos.
4. **Transfer learning** reuses a model that already "knows how to see", so it's fast and needs little data.
5. **Evaluate honestly** on a hidden test set, and **look at the mistakes**.

### 💬 Check your understanding
1. Why must all images be resized to the same size before training?
2. Why do we augment **training** images but **not** test images?
3. Our model was trained on bean leaves from Uganda. Would it work on **rice** leaves from Nueva Ecija? What would you need to do?

➡️ **Next: Activity 3: From Pixels to Predictions**, combining image processing and prediction **without** deep learning.